# Function calling: el contrato entre el modelo y tus herramientas

**Facsímil 5 · Agentes y orquestación** — capítulo 3 (*tools* y contratos operativos: *function
calling*).

Entre el modelo y tu base de datos (o tu cartera) hay una **frontera**. El modelo no «ejecuta» tus
funciones: **pide** ejecutarlas rellenando un formulario (nombre de la herramienta + argumentos), y tu
código decide si lo concede. *Function calling* es esa frontera, y es tan importante como lo fue la
separación entre datos y comandos para frenar la inyección SQL. En este cuaderno defines herramientas
con su **contrato**, aceptas una llamada bien hecha y **rechazas** una mal hecha *antes* de que cause
daño, pones **topes de negocio**, controlas los **campos de más**, añades **permisos**, conviertes el
error en un **contrato de reintento**, **auditas** cada llamada y, al final, miras el **esquema** que el
modelo recibe. Sin validación, una alucinación del modelo se convierte en una acción real.

### Qué vas a aprender
- Que el modelo **propone** una llamada (herramienta + argumentos) y tu código **dispone**.
- A declarar el **contrato** de cada herramienta con Pydantic: tipos, **enumerados** y **rangos**.
- A **validar** y rechazar las llamadas mal formadas con un motivo claro, sin tocar nada.
- A poner **topes de negocio** (no solo de tipo) y a controlar los **campos inesperados**.
- A poner **permisos**: qué ejecuta el agente solo y qué exige confirmación humana.
- A usar el **mensaje de error** como guía de reintento, a **auditar** las llamadas y a leer el **JSON
  Schema** que ve el modelo.

### Cuánto cuesta
Unos 16 minutos. CPU, sin claves (simulamos lo que pide el modelo).


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
# En Colab:  !pip install pydantic
from pydantic import BaseModel, Field, field_validator, ValidationError, ConfigDict
from typing import Literal

# Cada herramienta declara su CONTRATO: que argumentos acepta, de que tipo y con que limites.
class Convertir(BaseModel):
    cantidad: float = Field(gt=0, le=10_000)      # tope de negocio: nada de millones
    de: str
    a: str
    @field_validator("de", "a")
    @classmethod
    def moneda_valida(cls, v):
        if v.upper() not in {"EUR", "USD", "GBP"}:
            raise ValueError(f"moneda no soportada: {v}")
        return v.upper()

class Recordatorio(BaseModel):
    texto: str
    minutos: int
    @field_validator("minutos")
    @classmethod
    def futuro(cls, v):
        if v <= 0: raise ValueError("los minutos deben ser positivos")
        return v

CONTRATOS = {"convertir": Convertir, "recordatorio": Recordatorio}
TASAS = {("USD","EUR"): 0.92, ("EUR","USD"): 1.09, ("GBP","EUR"): 1.17}
print("Herramientas con contrato:", ", ".join(CONTRATOS))


## 1. El modelo propone; tú dispones

Cuando un modelo «usa una herramienta», en realidad devuelve un objeto: el **nombre** de la herramienta y
unos **argumentos** en JSON. No lo ejecutamos a ciegas. Lo pasamos por el **contrato** (el esquema de esa
herramienta). Si encaja, se ejecuta; si no, se rechaza con un motivo claro, **sin tocar nada**. Esta
separación —proponer frente a ejecutar— es lo que mantiene el control en tu lado, no en el del modelo.


In [ ]:
def ejecutar(nombre, args):
    if nombre == "convertir":
        t = TASAS.get((args.de, args.a))
        return f"{args.cantidad} {args.de} = {round(args.cantidad*t, 2)} {args.a}" if t else "par no disponible"
    if nombre == "recordatorio":
        return f"recordatorio '{args.texto}' en {args.minutos} min: creado"
    if nombre == "crear_tarea":
        return f"tarea '{args.titulo}' [{args.prioridad}, {args.horas} h]: creada"

def maneja(llamada):
    nombre, args_json = llamada["tool"], llamada["args"]
    if nombre not in CONTRATOS:
        return f"[RECHAZADA] herramienta desconocida: {nombre}"
    try:
        args = CONTRATOS[nombre].model_validate(args_json)    # valida contra el contrato
    except ValidationError as e:
        err = e.errors()[0]
        return f"[RECHAZADA] {nombre}: {err['loc'][0] if err['loc'] else '(raiz)'} -> {err['msg']}"
    return f"[OK] {ejecutar(nombre, args)}"

print("Llamadas BIEN hechas:")
for ll in [{"tool":"convertir","args":{"cantidad":100,"de":"usd","a":"eur"}},
           {"tool":"recordatorio","args":{"texto":"llamar al banco","minutos":30}}]:
    print(" ", maneja(ll))


## 2. Cuando el modelo alucina los argumentos

Los modelos, a veces, inventan una moneda, ponen un texto donde va un número o se dejan un campo. Sin
contrato, eso llega a tu lógica y revienta (o peor, hace algo). Con contrato, **se queda en la puerta**
con un motivo legible, antes de ejecutarse. Probamos cuatro alucinaciones típicas.


In [ ]:
malas = [
    ("moneda inventada",        {"tool":"convertir","args":{"cantidad":100,"de":"BTC","a":"eur"}}),
    ("minutos negativos",       {"tool":"recordatorio","args":{"texto":"x","minutos":-5}}),
    ("falta un campo",          {"tool":"recordatorio","args":{"texto":"falta el tiempo"}}),
    ("herramienta inexistente", {"tool":"borrar_todo","args":{}}),
]
for motivo, ll in malas:
    print(f"  {motivo:<24} -> {maneja(ll)}")


**Eso es el valor del contrato.** Cada llamada peligrosa se ha quedado en la puerta, con un motivo
legible, *antes* de ejecutarse. La moneda inventada, el tiempo imposible, el campo que falta y la
herramienta que no existe: ninguna ha llegado a tocar nada. El modelo propone; tu código es quien decide.
Es exactamente la misma idea que las salidas estructuradas del facsímil 4, aplicada a las **acciones** de
un agente.


## 3. Tipos ricos: enumerados y rangos

Un contrato no se limita a «esto es un número». Puede decir «esto es **uno de estos tres valores**» (un
enumerado) o «esto está **entre 0 y 40**» (un rango). Cuanto más expresivo es el contrato, menos basura
pasa. Añadimos una herramienta `crear_tarea` cuya **prioridad** solo admite `normal`, `urgente` o
`critica`, y cuyas **horas** estimadas deben caer en un rango razonable. Y, de paso, declaramos que no
acepta campos que no espera.


In [ ]:
class CrearTarea(BaseModel):
    model_config = ConfigDict(extra="forbid")          # cualquier campo de mas se rechaza
    titulo: str
    prioridad: Literal["normal", "urgente", "critica"]
    horas: float = Field(gt=0, le=40)

CONTRATOS["crear_tarea"] = CrearTarea

print("Bien:", maneja({"tool":"crear_tarea","args":{"titulo":"migrar base de datos","prioridad":"urgente","horas":8}}))
print("Prioridad fuera del enumerado:",
      maneja({"tool":"crear_tarea","args":{"titulo":"x","prioridad":"altisima","horas":2}}))
print("Horas fuera de rango:",
      maneja({"tool":"crear_tarea","args":{"titulo":"x","prioridad":"normal","horas":500}}))


**El enumerado y el rango hacen el trabajo.** «altisima» no es una prioridad válida y 500 horas no es
una estimación creíble: ambas se rechazan con un motivo claro, sin que tu lógica de negocio tenga que
escribir un solo `if`. El tipo *es* la validación. Cuanto más dices en el contrato, menos tienes que
defenderte después.


## 4. Topes de negocio, no solo de tipo

Que un argumento sea «un número válido» no significa que sea «un número **aceptable**». Convertir 100
euros es normal; convertir un millón, probablemente un error (o un ataque). Por eso `Convertir` declaró
`cantidad` con `gt=0` y `le=10000`: un tope de **negocio**, no solo de tipo. Lo comprobamos.


In [ ]:
print("Cantidad normal:  ", maneja({"tool":"convertir","args":{"cantidad":250,"de":"eur","a":"usd"}}))
print("Cantidad absurda: ", maneja({"tool":"convertir","args":{"cantidad":1_000_000,"de":"eur","a":"usd"}}))
print("Cantidad negativa:", maneja({"tool":"convertir","args":{"cantidad":-5,"de":"eur","a":"usd"}}))
print("\nEl tipo dice 'es un numero'; el tope de negocio dice 'es un numero que tiene sentido aqui'.")


**Los dos límites disparan.** El millón supera el tope y el negativo no llega al mínimo: los dos se
rechazan. Los topes de negocio son una red barata contra errores del modelo y contra peticiones
maliciosas. Ponerlos en el contrato es ponerlos **una vez**, en la frontera, en vez de repartir
comprobaciones por todo el código.


## 5. Campos de más: la política de estricto

¿Y si el modelo manda un campo que no existe en el contrato? Por defecto, Pydantic lo **ignora** en
silencio, lo cual a veces esconde un error (el modelo creía que ese campo hacía algo). Por eso `CrearTarea`
se declaró con `extra="forbid"`: cualquier campo inesperado se **rechaza**. Es una decisión de política;
estricto es más seguro cuando las acciones tienen consecuencias.


In [ ]:
extra = {"tool":"crear_tarea","args":{"titulo":"x","prioridad":"normal","horas":2,"asignar_a":"jefe"}}
print("Con campo de mas (extra='forbid'):", maneja(extra))

# Recordatorio NO declara extra='forbid', asi que el campo sobrante se ignora en silencio.
extra2 = {"tool":"recordatorio","args":{"texto":"reunion","minutos":15,"color":"rojo"}}
print("Sin esa politica (se ignora):     ", maneja(extra2))


**Dos políticas, dos comportamientos.** `crear_tarea` rechaza el campo sobrante; `recordatorio` lo
ignora y sigue. Ninguna es «la correcta» siempre: estricto pilla errores antes, permisivo tolera clientes
que mandan metadatos de más. Lo importante es **elegir** la política a conciencia, no heredarla por
descuido.


## 6. Permisos: no todo debe ser automático

Validar que una llamada está **bien formada** no es lo mismo que decidir que se puede **ejecutar sin
supervisión**. Convertir monedas es inofensivo; *enviar dinero* o *borrar una cuenta* no. Por eso los
agentes serios clasifican las herramientas por **riesgo** y exigen confirmación humana para las
peligrosas (capítulo 8). Añadimos una capa de permisos por encima de la validación.


In [ ]:
HERRAMIENTAS_SENSIBLES = {"enviar_dinero", "borrar_cuenta"}

def maneja_con_permisos(llamada, confirmado=False):
    nombre = llamada["tool"]
    if nombre in HERRAMIENTAS_SENSIBLES and not confirmado:
        return f"[PENDIENTE] '{nombre}' es una accion sensible: requiere confirmacion humana."
    if nombre in HERRAMIENTAS_SENSIBLES and confirmado:
        return f"[OK] '{nombre}' ejecutada (un humano lo autorizo)."
    return maneja(llamada)

env = {"tool":"enviar_dinero","args":{}}
print("Sin confirmar:", maneja_con_permisos(env, confirmado=False))
print("Confirmado:   ", maneja_con_permisos(env, confirmado=True))
print("Inofensiva:   ", maneja_con_permisos({"tool":"convertir","args":{"cantidad":50,"de":"eur","a":"usd"}}))
print("\nEl modelo puede PEDIR cualquier accion; los permisos deciden cual se ejecuta sola y cual no.")


**Validar y permitir son capas distintas.** Una llamada puede estar perfectamente formada y aun así
quedarse en `[PENDIENTE]` esperando a una persona. La validación protege de errores; los permisos
protegen de consecuencias. Las dos juntas son lo que te deja dar herramientas potentes a un agente sin
perder el sueño.


## 7. El mensaje de error como contrato de reintento

Un buen rechazo no solo protege: **enseña**. Si el motivo es legible, el propio modelo puede leerlo y
**corregir** en el siguiente intento, igual que tú corriges un formulario que te devuelven marcado en
rojo. Por eso los mensajes opacos («error 422») son tan malos: nadie sabe qué arreglar. Simulamos un
modelo que aluciona una moneda, lee el motivo y reintenta bien.


In [ ]:
def modelo_simulado(intento):
    # Intento 0: aluciona la moneda. Intento 1: corrige tras leer el motivo del rechazo.
    if intento == 0:
        return {"tool":"convertir","args":{"cantidad":100,"de":"BTC","a":"eur"}}
    return {"tool":"convertir","args":{"cantidad":100,"de":"usd","a":"eur"}}

r = maneja(modelo_simulado(0))
print("Intento 1:", r)
if r.startswith("[RECHAZADA]"):
    print("  -> el modelo lee el motivo y reintenta corrigiendo el argumento.")
    print("Intento 2:", maneja(modelo_simulado(1)))


**El bucle se cierra solo.** El primer intento se rechaza con un motivo concreto; el segundo, ya
corregido, pasa. Un mensaje de error claro es, en realidad, parte del contrato: le dice al modelo (o a
quien depure) exactamente qué cambiar. Invertir en errores legibles es invertir en que el sistema se
recupere sin intervención.


## 8. Auditoría: registrar cada llamada

En producción necesitas saber **qué pidió el agente**, no solo qué se ejecutó. Una llamada rechazada o
pendiente cuenta tanto como una aceptada para entender un incidente. Envolvemos el manejo en una capa que
**registra** cada intento con su resultado, y al final sacamos un resumen.


In [ ]:
from collections import Counter
REGISTRO = []

def maneja_auditado(llamada, confirmado=False):
    r = maneja_con_permisos(llamada, confirmado)
    etiqueta = r.split("]")[0].lstrip("[")     # OK / RECHAZADA / PENDIENTE
    REGISTRO.append({"tool": llamada["tool"], "resultado": etiqueta})
    return r

lote = [
    ({"tool":"convertir","args":{"cantidad":100,"de":"usd","a":"eur"}}, False),
    ({"tool":"convertir","args":{"cantidad":1_000_000,"de":"eur","a":"usd"}}, False),
    ({"tool":"enviar_dinero","args":{}}, False),
    ({"tool":"crear_tarea","args":{"titulo":"x","prioridad":"critica","horas":3}}, False),
    ({"tool":"borrar_todo","args":{}}, False),
]
for ll, conf in lote:
    maneja_auditado(ll, conf)

print(f"Se registraron {len(REGISTRO)} llamadas.")
for etiqueta, n in Counter(r["resultado"] for r in REGISTRO).most_common():
    print(f"  {etiqueta:<10}: {n}")
print("\nEsta lista es la auditoria del agente: la base para reconstruir que intento hacer.")


**Ahí está la trazabilidad.** Cinco intentos, clasificados por resultado: lo que se ejecutó, lo que se
rechazó y lo que quedó pendiente de un humano. Sin este registro, un incidente es un misterio; con él, es
una consulta. Auditar las **peticiones** del agente, y no solo sus efectos, es lo que permite explicar
después qué pasó y por qué.


## 9. El esquema que ve el modelo

Falta cerrar el círculo: ¿cómo sabe el modelo qué formulario rellenar? Le pasas el **esquema** de cada
herramienta, que Pydantic genera por ti en formato **JSON Schema**. Ese documento —nombres de campos,
tipos, enumerados, límites— es literalmente lo que el modelo recibe para saber cómo pedir la herramienta.
El contrato que tú escribes y la instrucción que el modelo lee son **el mismo objeto**.


In [ ]:
import json
print("Esquema de 'convertir' (esto es lo que recibe el modelo):\n")
print(json.dumps(Convertir.model_json_schema(), indent=2, ensure_ascii=False))


**Una sola fuente de verdad.** No hay un sitio donde defines el contrato y otro donde le explicas al
modelo cómo usarlo: el esquema sale del mismo `BaseModel` que valida las llamadas. Si añades un campo o un
límite, el modelo se entera y la validación lo exige, a la vez. Esa coherencia entre lo que el modelo
*sabe* y lo que tu código *exige* es justo lo que evita media clase de errores.


## 10. Pruébalo tú

1. **Un par de monedas nuevo.** Añade `("EUR","GBP")` a `TASAS` y comprueba que `convertir` de euros a
   libras ya funciona, sin tocar el contrato.
2. **Un enumerado más estricto.** Quita `"critica"` de la prioridad de `CrearTarea`. ¿Qué pasa ahora con
   una tarea crítica? Lee el motivo del rechazo.
3. **Cambia la política de campos.** Pon `extra="forbid"` también en `Recordatorio` y reenvía el
   recordatorio con `color`. Compara con el comportamiento de antes.
4. **Una herramienta sensible nueva.** Añade `cambiar_contrasena` a `HERRAMIENTAS_SENSIBLES` y compruébala
   con y sin `confirmado=True`.
5. **Audita un lote tuyo.** Mete tres llamadas más en `lote` (alguna mal formada) y mira cómo cambia el
   resumen del registro.


## 11. Errores comunes

- **Ejecutar lo que el modelo pide sin validar.** Una alucinación se convierte en una acción real. Valida
  *siempre* contra un contrato.
- **Quedarse en los tipos.** Un número válido puede ser un número absurdo; añade **topes de negocio**
  (rangos, enumerados) además de los tipos.
- **Ignorar los campos de más por descuido.** Decide tu política (`extra`) a conciencia; el silencio
  esconde errores del modelo.
- **Confundir «bien formado» con «permitido».** Una llamada puede ser válida y aun así requerir
  confirmación humana. Tipos y permisos son capas distintas.
- **Mensajes de error opacos.** Si el rechazo no dice qué falló, ni tú ni el modelo (en el reintento)
  podéis corregirlo.
- **No auditar.** Sin registro de qué herramientas pidió el agente (aceptadas, rechazadas y pendientes),
  es imposible entender un incidente.


## 12. Qué te llevas

- *Function calling* es un **contrato**: el modelo rellena un formulario (herramienta + argumentos) y tu
  código lo **valida antes de ejecutar**.
- Un buen contrato dice más que tipos: **enumerados**, **rangos** y **topes de negocio** dejan pasar menos
  basura, y la **política de campos** decide qué hacer con lo inesperado.
- La validación convierte una posible alucinación en un **rechazo limpio**, no en una acción real; el
  **mensaje de error** legible permite el **reintento**.
- Sobre la validación van los **permisos** (qué necesita un humano) y la **auditoría** (qué pidió el
  agente). Y todo nace del mismo esquema que el modelo **lee** para saber cómo pedir.

**Para seguir:** el capítulo 8 trata permisos y autonomía a fondo; el capítulo 10, cómo evaluar si el
agente, con todas sus herramientas, hace bien su trabajo; y el facsímil 9 conecta esto con la seguridad
(un texto externo que intenta forzar una llamada peligrosa: inyección de prompt).


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*